## Candidate vocabulary (15 words) — pending team approval

**Words:** hello, thankyou, please, yes, no, wait, finish, where, go, home, drink, book, airplane, car, jump
**Backups:** bye, give, read

**Checked on the full `train.csv`:**
- All 15 words exist with the exact spelling shown.
- Each word was performed by all 21 participants.
- 5,673 samples in total, 346 to 400 per word.
- Samples per participant (selected words only): 195 to 301.

**Checked on a small random sample (75 files, 5 per word, seed 42 — about 1.3% of the selected data):**
- Frames per sample: min 6, median 30, max 240. 37 of 75 samples have fewer than 30 frames.
- No sample without any detected hand. Frames with at least one hand detected: mean 0.69, median 0.72, min 0.11.
- Per word, the mean ratio of frames with a detected hand ranges from 0.43 (car) to 0.89 (please). With only 5 samples per word, these differences are not reliable and no word is removed because of them.
- Median frames per word range from 10 (jump) to 103 (wait): resampling to 30 frames will stretch some words and shrink others.

**Not checked yet:**
- Sign correctness and visual distinctiveness: to verify with a reliable ASL source (PM). Confusions between words will also show up in the confusion matrix (ML Engineer 2).
- The same measurements on all selected files (not just 75).
- Why hands are missing in some frames, and whether left/right labels match reality.
- Words signed mostly near the face were avoided because the contract does not use face landmarks (our assumption, not tested).



In [16]:
from pathlib import Path

import pandas as pd


TRAIN_CSV = Path(r"E:\SignSense_data\raw\train.csv")

df = pd.read_csv(TRAIN_CSV)

signs = (
    df.groupby("sign")
    .agg(samples=("sequence_id", "count"), participants=("participant_id", "nunique"))
    .sort_index()
)

print("Number of signs:", len(signs))
print(signs.to_string())

Number of signs: 250
              samples  participants
sign                               
TV                385            21
after             347            21
airplane          393            21
all               386            21
alligator         390            21
animal            348            21
another           370            21
any               355            21
apple             377            21
arm               362            21
aunt              391            21
awake             401            21
backyard          372            21
bad               376            21
balloon           393            21
bath              351            21
because           395            21
bed               353            21
bedroom           388            21
bee               395            21
before            354            21
beside            310            21
better            376            21
bird              404            21
black             388            21
blow   

In [17]:
SELECTED_WORDS = [
    "hello", "thankyou", "please", "yes", "no", "wait", "finish", "where",
    "go", "home", "drink", "book", "airplane", "car", "jump",
]
BACKUP_WORDS = ["bye", "give", "read"]

missing = [w for w in SELECTED_WORDS + BACKUP_WORDS if w not in signs.index]
print("Words not found in train.csv:", missing)

chosen = signs.loc[SELECTED_WORDS]
print(chosen.to_string())
print("\nTotal samples:", int(chosen["samples"].sum()))
print("Words seen by all 21 participants:", int((chosen["participants"] == 21).sum()), "of", len(chosen))

subset = df[df["sign"].isin(SELECTED_WORDS)]
print("\nSamples per participant (selected words only):")
print(subset["participant_id"].value_counts().describe())

Words not found in train.csv: []
          samples  participants
sign                           
hello         390            21
thankyou      357            21
please        384            21
yes           386            21
no            370            21
wait          346            21
finish        388            21
where         372            21
go            388            21
home          395            21
drink         400            21
book          368            21
airplane      393            21
car           353            21
jump          383            21

Total samples: 5673
Words seen by all 21 participants: 15 of 15

Samples per participant (selected words only):
count     21.000000
mean     270.142857
std       27.695281
min      195.000000
25%      266.000000
50%      282.000000
75%      286.000000
max      301.000000
Name: count, dtype: float64


In [18]:
sample = (
    df[df["sign"].isin(SELECTED_WORDS)]
    .groupby("sign")
    .sample(n=5, random_state=42)
)

out = Path(r"E:\SignSense_data\raw\sample_paths.txt")
out.write_text("\n".join(sample["path"]), encoding="utf-8")

print(len(sample), "paths saved to", out)
print(sample[["sign", "participant_id", "path"]].head())

75 paths saved to E:\SignSense_data\raw\sample_paths.txt
           sign  participant_id                                           path
15783  airplane           53618  train_landmark_files/53618/1651100440.parquet
65599  airplane           37779  train_landmark_files/37779/3689911218.parquet
58879  airplane           55372  train_landmark_files/55372/3408876773.parquet
11981  airplane           22343  train_landmark_files/22343/1493294633.parquet
93260  airplane           34503    train_landmark_files/34503/94974215.parquet


In [19]:
LANDMARKS_ROOT = Path(r"E:\SignSense_data\raw\landmarks_test")


def summarize_file(path):
    """Count frames and frames where each hand is detected (x not NaN)."""
    d = pd.read_parquet(path, columns=["frame", "type", "x"])
    n_frames = d["frame"].nunique()
    hands = d[d["type"].isin(["left_hand", "right_hand"])].dropna(subset=["x"])
    seen = hands.groupby("type")["frame"].nunique()
    return {
        "n_frames": n_frames,
        "left_frames": int(seen.get("left_hand", 0)),
        "right_frames": int(seen.get("right_hand", 0)),
        "any_hand_frames": hands["frame"].nunique(),
    }


rows = []
missing_files = []
for _, row in sample.iterrows():
    file_path = LANDMARKS_ROOT / row["path"]
    if not file_path.exists():
        missing_files.append(row["path"])
        continue
    info = summarize_file(file_path)
    info["sign"] = row["sign"]
    info["participant_id"] = row["participant_id"]
    rows.append(info)

quality = pd.DataFrame(rows)
quality["any_hand_ratio"] = quality["any_hand_frames"] / quality["n_frames"]

print("Files measured:", len(quality), "| missing:", len(missing_files))
print("\nFrames per sample:")
print(quality["n_frames"].describe())
print("\nSamples with fewer than 30 frames:", int((quality["n_frames"] < 30).sum()))
print("Samples with no hand detected at all:", int((quality["any_hand_frames"] == 0).sum()))
print("\nRatio of frames with at least one hand detected:")
print(quality["any_hand_ratio"].describe())

Files measured: 75 | missing: 0

Frames per sample:
count     75.000000
mean      47.146667
std       52.595523
min        6.000000
25%       16.500000
50%       30.000000
75%       48.000000
max      240.000000
Name: n_frames, dtype: float64

Samples with fewer than 30 frames: 37
Samples with no hand detected at all: 0

Ratio of frames with at least one hand detected:
count    75.000000
mean      0.691849
std       0.260487
min       0.111111
25%       0.500000
50%       0.718447
75%       0.934409
max       1.000000
Name: any_hand_ratio, dtype: float64


In [20]:
quality["left_ratio"] = quality["left_frames"] / quality["n_frames"]
quality["right_ratio"] = quality["right_frames"] / quality["n_frames"]

by_word = (
    quality.groupby("sign")
    .agg(
        median_frames=("n_frames", "median"),
        any_hand=("any_hand_ratio", "mean"),
        left=("left_ratio", "mean"),
        right=("right_ratio", "mean"),
    )
    .round(2)
    .sort_values("any_hand")
)
print(by_word.to_string())

          median_frames  any_hand  left  right
sign                                          
car                37.0      0.43  0.28   0.16
book               31.0      0.52  0.32   0.20
home               24.0      0.58  0.10   0.48
hello              40.0      0.59  0.26   0.33
jump               10.0      0.64  0.30   0.36
thankyou           20.0      0.65  0.20   0.45
yes                33.0      0.68  0.52   0.16
finish             31.0      0.70  0.11   0.59
no                 46.0      0.71  0.51   0.20
airplane           38.0      0.73  0.45   0.28
drink              16.0      0.79  0.20   0.59
where              34.0      0.79  0.18   0.61
wait              103.0      0.83  0.35   0.48
go                 20.0      0.84  0.37   0.48
please             30.0      0.89  0.49   0.40
